# Stage 1b — Full Labeling Run

**AAI-590 Capstone · Monish Yarapathineni**

Labels all ~20,602 unique (problem, wrong answer) pairs with the validated
multi-label formulation, producing the lookup table Stage 2 trains on.

Run top to bottom. **Sections 1–6 cost nothing** — Section 6 calibrates on a small
sample and projects the full cost, and the run is gated behind an explicit flag in
Section 7.

---

### What was settled before this

| | |
|---|---|
| Taxonomy | 8 categories, Eedi-derived (notebook 02) |
| Formulation | Multi-label, macro κ = 0.675 (notebook 04) |
| Pairs | 18,816 fill-in (top 10/problem) + 1,786 MC select-1 |
| Excluded | Select-all — a wrong subset can encode several errors at once |

### Design notes carried over from earlier runs

- **Checkpointed from the start.** This is a long run; Colab will disconnect.
  Progress is saved to Drive every `CHECKPOINT_EVERY` batches and the notebook
  resumes from it automatically.
- **Streaming.** The SDK refuses non-streaming requests whose `max_tokens` implies
  a possible >10 minute generation.
- **Adaptive batching.** A failed batch is split in half and retried rather than
  dropped, so a single bad batch does not leave a hole.
- **No size hint in the prompt.** An earlier version said "most reveal one or two"
  and the model reproduced that pattern mechanically.

---

## 1 · Setup and tunables

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    %pip install -q datasets huggingface_hub anthropic
except ImportError:
    IN_COLAB = False

import os, re, json, time, textwrap, random, math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
from anthropic import Anthropic

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (9, 4)})

RANDOM_SEED = 590
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

# ---- Tunables --------------------------------------------------------------
BATCH_SIZE       = 20        # larger than notebook 04: streaming + high max_tokens
MAX_TOKENS       = 24000
CHECKPOINT_EVERY = 10        # save every N batches
CALIB_BATCHES    = 5         # Section 6 — measures real token usage
RUN_FULL         = False     # Section 7 gate — set True after reading the estimate

CKPT_NAME = "stage1b_full_labels.json"
USE_DRIVE = True             # persist through runtime disconnects

print(f"Colab {IN_COLAB} | batch {BATCH_SIZE} | checkpoint every {CHECKPOINT_EVERY} batches")

### 1.1 · Mount Drive for checkpointing

A browser download only helps if you happen to be watching when it fires. Drive
survives a runtime crash.

In [ ]:
CKPT_PATH = CKPT_NAME

if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        os.makedirs('/content/drive/MyDrive/aai590', exist_ok=True)
        CKPT_PATH = '/content/drive/MyDrive/aai590/' + CKPT_NAME
        print("Checkpoint path:", CKPT_PATH)
    except Exception as e:
        print("Drive unavailable, falling back to local:", e)

print("Existing checkpoint:", os.path.exists(CKPT_PATH))

## 2 · Load taxonomy

In [ ]:
TAXONOMY_PATH = "misconception_taxonomy_v1.json"

if not os.path.exists(TAXONOMY_PATH):
    drive_tax = '/content/drive/MyDrive/aai590/' + TAXONOMY_PATH
    if os.path.exists(drive_tax):
        TAXONOMY_PATH = drive_tax
    elif IN_COLAB:
        from google.colab import files
        print("Upload misconception_taxonomy_v1.json:")
        up = files.upload()
        TAXONOMY_PATH = list(up.keys())[0]

with open(TAXONOMY_PATH) as f:
    artifact = json.load(f)

taxonomy       = artifact["taxonomy"]
categories     = taxonomy["categories"]
category_names = [c["name"] for c in categories]

print(f"{len(category_names)} categories loaded from {TAXONOMY_PATH}")
for n in category_names:
    print("  •", n)

## 3 · Load FoundationalASSIST

In [ ]:
if IN_COLAB:
    from huggingface_hub import login
    login()

HF_REPO = 'ASSISTments/FoundationalASSIST'
LOCAL_BASE = os.path.expanduser('~/Desktop/Educator/aai590-capstone/data/foundationalassist')

if IN_COLAB:
    problems_ds     = load_dataset(HF_REPO, 'Foundational ASSIST Dataset')
    interactions_ds = load_dataset(HF_REPO, 'Interactions')
else:
    problems_ds     = load_from_disk(os.path.join(LOCAL_BASE, 'Foundational ASSIST Dataset'))
    interactions_ds = load_from_disk(os.path.join(LOCAL_BASE, 'interactions'))

problems     = problems_ds['train'].to_pandas()
interactions = interactions_ds['train'].to_pandas()
print(f'Problems: {problems.shape}   Interactions: {interactions.shape}')

## 4 · Rebuild the labeling pairs

Same definition throughout: score of zero **and** submitted answer differs from
correct. Student column is `user_id`.

In [ ]:
def strip_html(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'xmlns="[^"]*"', '', text)
    text = re.sub(r'<mfrac>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</mfrac>', r'\1/\2', text)
    text = re.sub(r'<msup>\s*<mi>([^<]+)</mi>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<msup>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<mo>([^<]+)</mo>', r' \1 ', text)
    text = re.sub(r'<m[a-z]+>([^<]*)</m[a-z]+>', r'\1', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    for ent, ch in {'&nbsp;': ' ', '&lt;': '<', '&gt;': '>', '&amp;': '&',
                    '&le;': '≤', '&ge;': '≥', '&deg;': '°', '&times;': '×'}.items():
        text = text.replace(ent, ch)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'&[a-z]+;', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalise_answer(s):
    return "" if pd.isna(s) else str(s).strip().lower()


def norm_answer_key(s):
    if pd.isna(s):
        return ""
    s = re.sub(r'\s*,\s*', ' , ', str(s).strip())
    return re.sub(r'\s+', ' ', s)


merged = interactions.merge(
    problems[['problem_id', 'Problem Type', 'Answer Types', 'Fill-in Answers',
              'Multiple Choice Answers', 'Multiple Choice Options', 'Problem Body']],
    on='problem_id', how='left'
)

fillin_wrong = merged[
    (merged['Problem Type'] == 'Fill-in-the-blank(s)') &
    (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Fill-in Answers']), axis=1)
].copy()

mc1_wrong = merged[
    (merged['Problem Type'] == 'Multiple Choice (select 1)') &
    (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Multiple Choice Answers']), axis=1)
].copy()

fillin_wrong['answer_key'] = fillin_wrong['answer_text'].map(norm_answer_key)
mc1_wrong['answer_key']    = mc1_wrong['answer_text'].map(norm_answer_key)

print(f'Fill-in wrong rows : {len(fillin_wrong):,}')
print(f'MC-1 wrong rows    : {len(mc1_wrong):,}')

In [ ]:
TOP_N_FILLIN = 10

fillin_pairs = (fillin_wrong.groupby(['problem_id', 'answer_key'])
                .size().reset_index(name='n_students')
                .rename(columns={'answer_key': 'answer_text'}))
fillin_pairs['rank'] = (fillin_pairs.groupby('problem_id')['n_students']
                        .rank(method='first', ascending=False))
fillin_pairs = fillin_pairs[fillin_pairs['rank'] <= TOP_N_FILLIN]

mc_pairs = (mc1_wrong.groupby(['problem_id', 'answer_key'])
            .size().reset_index(name='n_students')
            .rename(columns={'answer_key': 'answer_text'}))

pmeta = problems.set_index('problem_id')

def build_triples(pairs, fmt):
    rows = []
    for _, r in pairs.iterrows():
        try:
            p = pmeta.loc[r['problem_id']]
        except KeyError:
            continue
        if isinstance(p, pd.DataFrame):
            p = p.iloc[0]
        correct = p['Fill-in Answers'] if fmt == 'fill-in' else p['Multiple Choice Answers']
        body = strip_html(p['Problem Body'])
        if not body or pd.isna(correct):
            continue
        rows.append({
            'problem_id': r['problem_id'], 'format': fmt,
            'problem_text': body, 'correct': strip_html(correct),
            'wrong': strip_html(r['answer_text']),
            'options': strip_html(p['Multiple Choice Options']) if fmt == 'mc' else '',
            'n_students': r['n_students'],
        })
    return pd.DataFrame(rows)

ALL = pd.concat([build_triples(fillin_pairs, 'fill-in'),
                 build_triples(mc_pairs, 'mc')], ignore_index=True)
ALL['key'] = ALL['problem_id'].astype(str) + '||' + ALL['wrong'].astype(str)
ALL = ALL.drop_duplicates('key').reset_index(drop=True)

print(f"\nTotal pairs to label : {len(ALL):,}   (design doc: ~20,602)")
print(ALL['format'].value_counts().to_string())
print(f"\nStudent attempts covered: {ALL['n_students'].sum():,}")

## 5 · Client and labeling machinery

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    api_key = os.environ.get("ANTHROPIC_API_KEY")

assert api_key, "ANTHROPIC_API_KEY not found — add it to Colab secrets."
client = Anthropic(api_key=api_key)
LABELING_MODEL = "claude-sonnet-5"

PRICE_IN, PRICE_OUT = 3.0, 15.0      # USD per million tokens


def response_text(resp):
    parts = [b.text for b in resp.content if getattr(b, "type", None) == "text"]
    if not parts:
        raise ValueError(f"No text block; blocks={[getattr(b,'type','?') for b in resp.content]}")
    return "".join(parts)


def extract_json(text):
    cand = None
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.S)
    if fenced:
        cand = fenced.group(1)
        try:
            return json.loads(cand)
        except json.JSONDecodeError:
            cand = None
    if cand is None:
        try:
            cand = text[text.index("{"):text.rindex("}") + 1]
        except ValueError:
            raise ValueError(f"No JSON found:\n{text[:600]}")
    try:
        return json.loads(cand)
    except json.JSONDecodeError:
        return json.loads(re.sub(r"}\s*{", "},{", cand))


def as_list(v):
    if isinstance(v, str):
        v = [v]
    return [x for x in (v or []) if x in category_names or x == 'UNASSIGNABLE']


MULTI_TEMPLATE = """You are labeling student wrong answers with the misconception \
type(s) they reveal.

CATEGORIES:
{cats}

For each numbered item you get the problem, the correct answer, and what the \
student submitted.

Rules:
- Select every category that genuinely applies to this specific wrong answer. \
Some answers reveal a single clear misconception; others are consistent with \
several at once. Judge each item on its own evidence.
- Include a category only if you would defend it against a colleague. Do not add \
categories to hedge, and do not omit a category that genuinely fits.
- There is no expected number of labels. Return as many or as few as the evidence \
supports.
- If none genuinely applies, return ["UNASSIGNABLE"].
- Order most to least likely.

Return ONLY a JSON object mapping each item number to an array of category names.
Example shows format only — the number of entries carries no meaning:
{{"1": [...], "2": [...], "3": [...]}}

ITEMS:
{items}"""

cats_block = "\n".join(f"- {c['name']}: {c['definition']}" for c in categories)


def format_item(i, r):
    lines = [f"{i}. PROBLEM: {textwrap.shorten(r['problem_text'], 400)}"]
    if r["options"]:
        lines.append(f"   OPTIONS: {textwrap.shorten(r['options'], 300)}")
    lines.append(f"   CORRECT ANSWER: {r['correct']}")
    lines.append(f"   STUDENT SUBMITTED: {r['wrong']}")
    return "\n".join(lines)


USAGE = {"in": 0, "out": 0, "calls": 0}


def label_batch(df_batch, max_retries=2):
    """Streaming call over a batch. Returns {key: [labels]}."""
    items = "\n\n".join(format_item(i + 1, r)
                         for i, (_, r) in enumerate(df_batch.iterrows()))
    keys = list(df_batch['key'])
    last = None
    for attempt in range(max_retries):
        try:
            with client.messages.stream(
                model=LABELING_MODEL,
                max_tokens=MAX_TOKENS,
                messages=[{"role": "user", "content":
                           MULTI_TEMPLATE.format(cats=cats_block, items=items)}],
            ) as stream:
                final = stream.get_final_message()
            if final.stop_reason == "max_tokens":
                raise ValueError(f"hit max_tokens on batch of {len(df_batch)}")
            USAGE["in"]  += final.usage.input_tokens
            USAGE["out"] += final.usage.output_tokens
            USAGE["calls"] += 1
            parsed = extract_json(response_text(final))
            return {keys[i]: as_list(parsed.get(str(i + 1)))
                    for i in range(len(keys)) if parsed.get(str(i + 1))}
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last


def label_with_checkpoint(df, ckpt_path, batch_size, checkpoint_every, tag=""):
    """Resumable labeling. Skips keys already present in the checkpoint."""
    labels = {}
    if os.path.exists(ckpt_path):
        with open(ckpt_path) as f:
            labels = json.load(f).get("labels", {})
        print(f"  resumed from checkpoint: {len(labels):,} pairs already labeled")

    todo = df[~df['key'].isin(labels)].reset_index(drop=True)
    print(f"  remaining to label: {len(todo):,}")
    if len(todo) == 0:
        return labels, []

    failed, t0 = [], time.time()
    n_batches = math.ceil(len(todo) / batch_size)

    def save():
        tmp = ckpt_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"model": LABELING_MODEL,
                       "updated_utc": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                       "n": len(labels), "usage": USAGE, "labels": labels}, f)
        os.replace(tmp, ckpt_path)

    for b in range(n_batches):
        chunk = todo.iloc[b * batch_size:(b + 1) * batch_size]
        try:
            labels.update(label_batch(chunk))
        except Exception as e:
            if len(chunk) > 1:
                mid = len(chunk) // 2
                for sub in (chunk.iloc[:mid], chunk.iloc[mid:]):
                    try:
                        labels.update(label_batch(sub))
                    except Exception as e2:
                        failed.extend(sub['key'].tolist())
                        print(f"\n  {tag} sub-batch failed: {e2}")
            else:
                failed.extend(chunk['key'].tolist())
                print(f"\n  {tag} item failed: {e}")

        if (b + 1) % checkpoint_every == 0 or b == n_batches - 1:
            save()

        done = min((b + 1) * batch_size, len(todo))
        el = time.time() - t0
        rate = done / el if el else 0
        eta = (len(todo) - done) / rate if rate else 0
        cost = USAGE["in"] / 1e6 * PRICE_IN + USAGE["out"] / 1e6 * PRICE_OUT
        print(f"\r  {tag} {done:,}/{len(todo):,}  "
              f"({rate:.1f}/s, ETA {eta/60:.0f}m, ${cost:.2f})", end="")

    save()
    print()
    return labels, failed

## 6 · Calibration — measure real cost before committing

Extended thinking is billed as output, so a token estimate made on paper will be
badly wrong. This runs a handful of batches, measures actual usage, and
extrapolates.

Costs roughly `CALIB_BATCHES × BATCH_SIZE` pairs — a few cents.

In [ ]:
calib = ALL.sample(CALIB_BATCHES * BATCH_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

USAGE.update({"in": 0, "out": 0, "calls": 0})
calib_labels, calib_failed = {}, []
t0 = time.time()
for b in range(CALIB_BATCHES):
    chunk = calib.iloc[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    try:
        calib_labels.update(label_batch(chunk))
    except Exception as e:
        calib_failed.append(b); print(f"\n  calib batch {b} failed: {e}")
    print(f"\r  calibrating {min((b+1)*BATCH_SIZE, len(calib))}/{len(calib)}", end="")
elapsed = time.time() - t0
print()

per_call_in  = USAGE["in"] / max(USAGE["calls"], 1)
per_call_out = USAGE["out"] / max(USAGE["calls"], 1)
calls_needed = math.ceil(len(ALL) / BATCH_SIZE)
proj_in  = per_call_in * calls_needed
proj_out = per_call_out * calls_needed
proj_cost = proj_in / 1e6 * PRICE_IN + proj_out / 1e6 * PRICE_OUT
proj_time = elapsed / max(USAGE["calls"], 1) * calls_needed

print(f"\n{'=' * 70}")
print("CALIBRATION")
print("=" * 70)
print(f"  calls made            : {USAGE['calls']}")
print(f"  pairs labeled         : {len(calib_labels)} / {len(calib)}")
print(f"  mean labels per pair  : {np.mean([len(v) for v in calib_labels.values()]):.2f}")
print(f"  input tokens / call   : {per_call_in:,.0f}")
print(f"  output tokens / call  : {per_call_out:,.0f}   (includes thinking)")
print(f"  seconds / call        : {elapsed/max(USAGE['calls'],1):.1f}")
print()
print("PROJECTED FULL RUN")
print("-" * 70)
print(f"  pairs                 : {len(ALL):,}")
print(f"  calls at batch {BATCH_SIZE}     : {calls_needed:,}")
print(f"  input tokens          : {proj_in/1e6:,.1f}M")
print(f"  output tokens         : {proj_out/1e6:,.1f}M")
print(f"  ESTIMATED COST        : ${proj_cost:,.2f}")
print(f"  ESTIMATED WALL TIME   : {proj_time/60:,.0f} min")
print("=" * 70)
print("  Design doc budgeted ~$30, before extended thinking was billed as output.")
print("  If this projection is uncomfortable, raise BATCH_SIZE (fewer calls means")
print("  the category block is resent less often and thinking amortises further),")
print("  then re-run this cell.")

## 7 · Full run

**Gated.** Set `RUN_FULL = True` in Section 1 and re-run that cell first.

Safe to interrupt — progress is checkpointed to Drive every
`CHECKPOINT_EVERY` batches, and re-running this cell resumes from where it
stopped.

In [ ]:
if not RUN_FULL:
    print("RUN_FULL is False — skipping.")
    print("Set RUN_FULL = True in Section 1, re-run that cell, then run this one.")
else:
    USAGE.update({"in": 0, "out": 0, "calls": 0})
    print(f"Labeling {len(ALL):,} pairs")
    all_labels, failed_keys = label_with_checkpoint(
        ALL, CKPT_PATH, BATCH_SIZE, CHECKPOINT_EVERY, tag="[full]")

    cost = USAGE["in"] / 1e6 * PRICE_IN + USAGE["out"] / 1e6 * PRICE_OUT
    print(f"\nLabeled     : {len(all_labels):,} / {len(ALL):,}")
    print(f"Failed keys : {len(failed_keys):,}")
    print(f"Actual cost : ${cost:,.2f}  ({USAGE['calls']:,} calls)")

### 7.1 · Retry any stragglers

In [ ]:
if RUN_FULL and 'failed_keys' in dir() and failed_keys:
    retry = ALL[ALL['key'].isin(failed_keys)].reset_index(drop=True)
    print(f"Retrying {len(retry):,} failed pairs at batch size 5")
    more, still_failed = label_with_checkpoint(retry, CKPT_PATH, 5, 5, tag="[retry]")
    all_labels.update(more)
    print(f"Recovered {len(more):,}; still failing {len(still_failed):,}")
elif RUN_FULL:
    print("No failures to retry.")
else:
    print("Skipped (RUN_FULL is False).")

## 8 · Build the Stage 2 lookup artifact

In [ ]:
with open(CKPT_PATH) as f:
    ck = json.load(f)
all_labels = ck["labels"]

lookup = ALL[['problem_id', 'wrong', 'format', 'n_students', 'key']].copy()
lookup['labels'] = lookup['key'].map(all_labels)
lookup['n_labels'] = lookup['labels'].apply(lambda v: len(v) if isinstance(v, list) else 0)
labeled = lookup[lookup['n_labels'] > 0]

print(f"Pairs total    : {len(lookup):,}")
print(f"Pairs labeled  : {len(labeled):,}  ({len(labeled)/len(lookup):.1%})")
print(f"Mean labels    : {labeled['n_labels'].mean():.2f}")
print()
print("Set size distribution:")
for k, v in sorted(Counter(labeled['n_labels']).items()):
    print(f"  {k}: {v:>6,}  ({v/len(labeled):>5.1%})")

# Attempt-weighted category frequencies — what pos_weight should be derived from
cat_pairs, cat_attempts = Counter(), Counter()
for _, r in labeled.iterrows():
    for l in r['labels']:
        if l in category_names:
            cat_pairs[l] += 1
            cat_attempts[l] += r['n_students']

tot_att = sum(cat_attempts.values())
print(f"\n{'Category':<36}{'pairs':>9}{'attempts':>12}{'share':>8}")
print("-" * 66)
for cname in category_names:
    print(f"{cname[:34]:<36}{cat_pairs.get(cname,0):>9,}"
          f"{cat_attempts.get(cname,0):>12,}{cat_attempts.get(cname,0)/tot_att:>8.1%}")

In [ ]:
OUT_JSON = "misconception_labels_v1.json"

records = [{"problem_id": int(r['problem_id']), "wrong": r['wrong'],
            "format": r['format'], "n_students": int(r['n_students']),
            "labels": r['labels']}
           for _, r in labeled.iterrows()]

payload = {
    "generated_utc": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    "model": LABELING_MODEL,
    "formulation": "multi-label",
    "taxonomy_source": artifact.get("source_corpus"),
    "categories": category_names,
    "n_pairs_total": int(len(lookup)),
    "n_pairs_labeled": int(len(labeled)),
    "mean_labels_per_pair": float(labeled['n_labels'].mean()),
    "category_pair_counts": dict(cat_pairs),
    "category_attempt_counts": dict(cat_attempts),
    "records": records,
}

with open(OUT_JSON, "w") as f:
    json.dump(payload, f)
print(f"Wrote {OUT_JSON}  ({os.path.getsize(OUT_JSON)/1e6:.1f} MB)")

if IN_COLAB and USE_DRIVE:
    try:
        import shutil
        shutil.copy(OUT_JSON, '/content/drive/MyDrive/aai590/' + OUT_JSON)
        print("Copied to Drive/MyDrive/aai590/")
    except Exception as e:
        print("Drive copy skipped:", e)

if IN_COLAB:
    from google.colab import files
    files.download(OUT_JSON)

## 9 · Coverage report

How much of the 605,773 misconception rows now carry a label. The remainder are
masked in the Stage 2 loss — they stay in the student sequence as behavioural
context but contribute no gradient.

In [ ]:
fw = fillin_wrong.copy(); fw['wrong'] = fw['answer_key'].map(strip_html)
mw = mc1_wrong.copy();    mw['wrong'] = mw['answer_key'].map(strip_html)
fw['key'] = fw['problem_id'].astype(str) + '||' + fw['wrong'].astype(str)
mw['key'] = mw['problem_id'].astype(str) + '||' + mw['wrong'].astype(str)

labeled_keys = set(labeled['key'])
fw_cov = fw['key'].isin(labeled_keys).mean()
mw_cov = mw['key'].isin(labeled_keys).mean()

n_fill, n_mc = len(fw), len(mw)
n_selectall = 119714     # measured in notebook 03
total_misc = n_fill + n_mc + n_selectall

covered = fw['key'].isin(labeled_keys).sum() + mw['key'].isin(labeled_keys).sum()

print("COVERAGE OF MISCONCEPTION ROWS")
print("=" * 70)
print(f"  fill-in rows          : {n_fill:>9,}   labeled {fw_cov:>6.1%}")
print(f"  MC select-1 rows      : {n_mc:>9,}   labeled {mw_cov:>6.1%}")
print(f"  select-all rows       : {n_selectall:>9,}   labeled   0.0%  (excluded by design)")
print("-" * 70)
print(f"  total misconception   : {total_misc:>9,}")
print(f"  with a label          : {covered:>9,}   ({covered/total_misc:.1%})")
print(f"  masked in Stage 2     : {total_misc-covered:>9,}   ({1-covered/total_misc:.1%})")
print("=" * 70)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

order = sorted(category_names, key=lambda c: -cat_attempts.get(c, 0))
axes[0].barh([c[:26] for c in order][::-1],
             [cat_attempts.get(c, 0) for c in order][::-1], color='#2c5f8a')
axes[0].set_xlabel('student attempts')
axes[0].set_title('Category frequency (attempt-weighted)', fontsize=11)
axes[0].tick_params(labelsize=8)

sizes = Counter(labeled['n_labels'])
axes[1].bar(list(sizes.keys()), list(sizes.values()), color='#6b9dbf')
axes[1].set_xlabel('labels per pair'); axes[1].set_ylabel('pairs')
axes[1].set_title(f"Label-set size (mean {labeled['n_labels'].mean():.2f})", fontsize=11)

plt.tight_layout()
plt.savefig('stage1b_full_run.png', dpi=200, bbox_inches='tight')
plt.show()

---

## Next steps

1. Commit `misconception_labels_v1.json` to `aai590-capstone/labels/`.
2. Record the actual cost, coverage, and category frequencies in
   `capstone/stage1-pilot-findings.md`.
3. Notebook 06 — join labels onto interactions, engineer features, and train the
   LSTM with 8 sigmoid outputs and BCE loss, `pos_weight` derived from the
   attempt-weighted counts above.
4. Build the **context-free baseline** alongside it. Whether sequence context
   improves per-interaction prediction is the hypothesis; the baseline is what
   makes that claim testable.

---

*02 taxonomy · 03 single-label pilot · 04 multi-label validation · 05 full run.*